# Explore EMIT Granule Structure

This notebook opens one set of already-downloaded EMIT granules (L1B RAD,
L1B OBS, L2A RFL) and inspects their internal structure. The primary goal
is to understand the OBS file's band layout so we can unambiguously identify
the Solar Zenith Angle (SZA) and View Zenith Angle (VZA) band indices.

**Prerequisites:** Run `scripts/run_extraction.py` first to download at least
one granule set into `data/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import netCDF4

## 1. Locate granule files in data/

We look for NetCDF4 files matching EMIT product naming patterns.
If nothing is found, we print a message and stop gracefully.

In [ ]:
DATA_DIR = Path("..") / "data"

# EMIT files follow patterns like:
#   EMIT_L1B_RAD_*.nc
#   EMIT_L1B_OBS_*.nc
#   EMIT_L2A_RFL_*.nc
# Search recursively — extraction may place files in subdirectories.

def find_first(pattern: str) -> Path | None:
    """Return the first file matching a glob pattern under DATA_DIR."""
    matches = sorted(DATA_DIR.rglob(pattern))
    return matches[0] if matches else None

rad_file = find_first("*L1B_RAD*.nc")
obs_file = find_first("*L1B_OBS*.nc")
rfl_file = find_first("*L2A_RFL*.nc")

found = {"L1B_RAD": rad_file, "L1B_OBS": obs_file, "L2A_RFL": rfl_file}

for label, path in found.items():
    if path:
        print(f"{label}: {path}")
    else:
        print(f"{label}: NOT FOUND")

if not any(found.values()):
    print(
        "\n*** No granule files found in data/. ***\n"
        "Run the extraction script first:\n"
        "  python scripts/run_extraction.py --site libya_4 "
        "--start 2023-01-01 --end 2023-12-31\n"
    )

## 2. Inspect each file with xarray

EMIT NetCDF4 files use groups. The root group typically has the main data
variable, while metadata (wavelengths, band parameters) lives in subgroups.
We open both the root and any subgroups to see the full picture.

In [ ]:
def inspect_nc(filepath: Path) -> None:
    """Print the structure of an EMIT NetCDF4 file: root + all groups."""
    if filepath is None:
        print("(file not available, skipping)\n")
        return

    print(f"=== {filepath.name} ===")
    print(f"    Size: {filepath.stat().st_size / 1e6:.1f} MB\n")

    # --- Root group via xarray ---
    ds = xr.open_dataset(filepath, engine="netcdf4")
    print("-- Root group (xarray) --")
    print(ds)
    print()

    # --- List all groups via netCDF4 ---
    with netCDF4.Dataset(str(filepath)) as nc:
        groups = list(nc.groups.keys())
        print(f"Groups: {groups}")

        # Print global attributes
        if nc.ncattrs():
            print("\nGlobal attributes:")
            for attr in nc.ncattrs():
                val = nc.getncattr(attr)
                # Truncate long values for readability
                val_str = str(val)
                if len(val_str) > 120:
                    val_str = val_str[:120] + "..."
                print(f"    {attr}: {val_str}")

        # Open each subgroup
        for gname in groups:
            print(f"\n-- Group: {gname} --")
            gds = xr.open_dataset(filepath, engine="netcdf4", group=gname)
            print(gds)

    print("\n" + "=" * 60 + "\n")

In [ ]:
# Inspect RAD file
inspect_nc(rad_file)

In [ ]:
# Inspect OBS file
inspect_nc(obs_file)

In [ ]:
# Inspect RFL file
inspect_nc(rfl_file)

## 3. OBS band layout — the key mapping

The OBS file stores observation geometry as a 3-D array:
`(downtrack, crosstrack, bands)`. The bands are **not labeled** in the
main data variable — their names/descriptions live in a metadata group
(typically `sensor_band_parameters` or `observation_bands`).

We extract that metadata here and print a clean index-to-variable mapping
so the SZA and VZA indices are unambiguous.

In [ ]:
if obs_file is None:
    print("OBS file not found — cannot extract band mapping.")
else:
    # Open with netCDF4 to inspect groups and variables directly.
    with netCDF4.Dataset(str(obs_file)) as nc:
        print("OBS root variables:", list(nc.variables.keys()))
        print("OBS groups:", list(nc.groups.keys()))
        print()

        # The band names are typically in the 'sensor_band_parameters' group
        # under a variable like 'observation_bands'.
        # We search common locations.
        band_names = None

        # Check sensor_band_parameters group
        if "sensor_band_parameters" in nc.groups:
            sbp = nc.groups["sensor_band_parameters"]
            print("sensor_band_parameters variables:", list(sbp.variables.keys()))
            for vname in sbp.variables:
                var = sbp.variables[vname]
                data = var[:]
                print(f"  {vname}: shape={data.shape}, dtype={data.dtype}")
                # If it's a string array, these are the band names
                if data.dtype.kind in ('U', 'S', 'O'):
                    band_names = [str(s) for s in data]
                    print(f"    -> values: {band_names}")
                else:
                    print(f"    -> values: {data}")
                print()

        # Fallback: check root attributes or other groups
        if band_names is None:
            print("Did not find band names in sensor_band_parameters.")
            print("Checking root variable attributes...")
            for vname in nc.variables:
                var = nc.variables[vname]
                print(f"  {vname}: dims={var.dimensions}, shape={var.shape}")
                for attr in var.ncattrs():
                    print(f"    @{attr} = {var.getncattr(attr)}")

    # Print the clean mapping
    if band_names:
        print("\n" + "=" * 50)
        print("OBS BAND INDEX MAPPING")
        print("=" * 50)
        for i, name in enumerate(band_names):
            marker = ""
            name_lower = name.lower()
            if "sun" in name_lower and "zenith" in name_lower:
                marker = "  <-- SZA"
            elif "sensor" in name_lower and "zenith" in name_lower:
                marker = "  <-- VZA"
            elif "view" in name_lower and "zenith" in name_lower:
                marker = "  <-- VZA"
            print(f"  Band {i:2d}: {name}{marker}")
        print("=" * 50)
    else:
        print(
            "\nCould not automatically find band names. "
            "Check the output above and update this notebook."
        )

## 4. SZA / VZA sanity check

Pull the SZA and VZA bands by index, print summary stats, and plot one
band as a raw (non-orthorectified) 2-D array.

**Assumption:** The expected band indices (based on EMIT documentation) are:
- SZA = band 4 (`to_sun_zenith`)
- VZA = band 2 (`to_sensor_zenith`)

Verify these against the mapping printed in the cell above and update
if they differ.

In [ ]:
# Indices based on expected EMIT OBS band layout.
# UPDATE these if the mapping above shows different positions.
SZA_BAND_INDEX = 4  # to_sun_zenith
VZA_BAND_INDEX = 2  # to_sensor_zenith

In [ ]:
if obs_file is None:
    print("OBS file not found — skipping.")
else:
    ds_obs = xr.open_dataset(obs_file, engine="netcdf4")

    # The main observation variable — find it by looking for the 3-D array.
    obs_var_name = None
    for vname in ds_obs.data_vars:
        if ds_obs[vname].ndim == 3:
            obs_var_name = vname
            break

    if obs_var_name is None:
        print("Could not find a 3-D observation variable. Check the file structure above.")
    else:
        print(f"Using variable: '{obs_var_name}', shape: {ds_obs[obs_var_name].shape}")
        obs_data = ds_obs[obs_var_name].values  # (downtrack, crosstrack, bands)

        sza = obs_data[:, :, SZA_BAND_INDEX]
        vza = obs_data[:, :, VZA_BAND_INDEX]

        # Mask fill values (typically -9999 or similar)
        sza = np.where(sza < -900, np.nan, sza)
        vza = np.where(vza < -900, np.nan, vza)

        print("\nSolar Zenith Angle (SZA):")
        print(f"  min  = {np.nanmin(sza):.2f}")
        print(f"  max  = {np.nanmax(sza):.2f}")
        print(f"  mean = {np.nanmean(sza):.2f}")

        print("\nView Zenith Angle (VZA):")
        print(f"  min  = {np.nanmin(vza):.2f}")
        print(f"  max  = {np.nanmax(vza):.2f}")
        print(f"  mean = {np.nanmean(vza):.2f}")

In [ ]:
if obs_file is not None and obs_var_name is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    im0 = axes[0].imshow(sza, aspect="auto", cmap="viridis")
    axes[0].set_title(f"SZA (band {SZA_BAND_INDEX}) — raw swath")
    axes[0].set_xlabel("Crosstrack")
    axes[0].set_ylabel("Downtrack")
    plt.colorbar(im0, ax=axes[0], label="degrees")

    im1 = axes[1].imshow(vza, aspect="auto", cmap="viridis")
    axes[1].set_title(f"VZA (band {VZA_BAND_INDEX}) — raw swath")
    axes[1].set_xlabel("Crosstrack")
    axes[1].set_ylabel("Downtrack")
    plt.colorbar(im1, ax=axes[1], label="degrees")

    plt.tight_layout()
    plt.show()
else:
    print("Skipping plot — OBS data not available.")

## Notes

- All arrays above are in the **raw (non-orthorectified)** pixel grid.
  To get geographic coordinates, the GLT must be applied (see `emit_pics.extract.apply_glt`).
- The VZA pattern across the crosstrack direction should show a characteristic
  bow-tie or symmetric increase from nadir — a good sanity check.
- If the band mapping above doesn't match expectations, the `sensor_band_parameters`
  group output is the authoritative source. Update `SZA_BAND_INDEX` and
  `VZA_BAND_INDEX` accordingly.